In [1]:
import pandas as pd
#from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
import numpy as np
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

In [3]:
model_df = pd.read_csv("../data/processed/lastfm_scrobbles_clean_tags_final_tfid.csv")

In [6]:
model_df.head()

,artist,track,timestamp,year_file,artist_clean,track_clean,parentheses,tags,tags_clean,tags_normalized,tags_filtered,tags_str
0,Oasis,Don't Look Back in Anger,1199052426,2007,oasis,dont look back in anger,[],['britpop' 'rock' 'british' 'alternative' 'ind...,['indie' 'rock' 'british' '90s' 'alternative r...,"['indie', 'rock', 'british', '90s', 'alternati...","['indie', 'alternative', 'britpop']","[ ' i n d i e ' , _ ' a l t e r n a t i v e ' ..."
1,Oasis,Wonderwall,1199052167,2007,oasis,wonderwall,[],['britpop' 'rock' 'british' 'alternative' 'ind...,['indie' 'rock' 'british' '90s' 'alternative r...,"['indie', 'rock', 'british', '90s', 'alternati...","['indie', 'alternative', 'britpop']","[ ' i n d i e ' , _ ' a l t e r n a t i v e ' ..."
2,Incubus,Aqueous Transmission,1199051161,2007,incubus,aqueous transmission,[],['alternative rock' 'rock' 'alternative' 'indi...,['american' 'indie' 'rock' 'alternative rock' ...,"['american', 'indie', 'rock', 'funk metal', 'a...","['indie', 'funk metal', 'alternative', 'groovy...","[ ' i n d i e ' , _ ' f u n k _ m e t a l ' , ..."
3,Incubus,Under My Umbrella12DSGMS192,1199050947,2007,incubus,under my umbrella,[],['alternative rock' 'rock' 'alternative' 'indi...,['american' 'indie' 'rock' 'alternative rock' ...,"['american', 'indie', 'rock', 'funk metal', 'a...","['indie', 'funk metal', 'alternative', 'groovy...","[ ' i n d i e ' , _ ' f u n k _ m e t a l ' , ..."
4,Incubus,Are You In11DSGMS192,1199050681,2007,incubus,are you in,[],['alternative rock' 'rock' 'alternative' 'indi...,['american' 'indie' 'rock' 'alternative rock' ...,"['american', 'indie', 'rock', 'funk metal', 'a...","['indie', 'funk metal', 'alternative', 'groovy...","[ ' i n d i e ' , _ ' f u n k _ m e t a l ' , ..."


In [8]:
import ast

model_df["tags_filtered"] = model_df["tags_filtered"].apply(ast.literal_eval)

In [9]:
#TFIDFVectorizer breaks tags with spaces, so we need to join them with underscores

def join_tags(tags):
    return " ".join([t.replace(" ", "_") for t in tags])

model_df["tags_str"] = model_df["tags_filtered"].apply(join_tags)

In [10]:
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(model_df["tags_str"])

In [11]:
model_df["tags_str"].head(10)

0                         indie alternative britpop
1                         indie alternative britpop
2    indie funk_metal alternative groovy jecks funk
3    indie funk_metal alternative groovy jecks funk
4    indie funk_metal alternative groovy jecks funk
5    indie funk_metal alternative groovy jecks funk
6    indie funk_metal alternative groovy jecks funk
7    indie funk_metal alternative groovy jecks funk
8    indie funk_metal alternative groovy jecks funk
9    indie funk_metal alternative groovy jecks funk
Name: tags_str, dtype: object

In [ ]:
# mlb = MultiLabelBinarizer()
# X = mlb.fit_transform(model_df["tags_final"])

In [12]:
model_df["tags_str"][5]

'indie funk_metal alternative groovy jecks funk'

In [13]:
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=50, random_state=42)
X_reduced = svd.fit_transform(X)

In [14]:
kmeans = KMeans(n_clusters=8, random_state=42, n_init=20)
clusters = kmeans.fit_predict(X_reduced)
model_df["cluster"] = clusters

In [15]:
feature_names = vectorizer.get_feature_names_out()

def get_top_tags_per_cluster(X, clusters, feature_names, top_n=10):
    cluster_tags = {}
    
    for cluster in np.unique(clusters):
        idx = clusters == cluster
        mean_vals = np.asarray(X[idx].mean(axis=0)).ravel()
        top_idx = np.argsort(mean_vals)[-top_n:]
        cluster_tags[cluster] = [feature_names[i] for i in top_idx]
        
    return cluster_tags

In [16]:
top_tags = get_top_tags_per_cluster(X, clusters, feature_names)

for k, v in top_tags.items():
    print(f"Cluster {k}: {v}")

Cluster 0: ['gothic', 'goth', 'classic_rock', 'gothic_rock', 'indie', 'post_punk_revival', 'synthpop', 'alternative', 'post_punk', 'new_wave']
Cluster 1: ['space_rock', 'art_rock', 'garage_rock', 'indie_pop', 'post_punk', 'post_punk_revival', 'glam_rock', 'indie', 'alternative', 'britpop']
Cluster 2: ['lo_fi', 'art_rock', 'folk', 'synthpop', 'electropop', 'art_pop', 'experimental', 'alternative', 'indie', 'indie_pop']
Cluster 3: ['blues', 'punk', 'instrumental', 'jazz', 'soul', 'ambient', 'classic_rock', 'indie', 'experimental', 'alternative']
Cluster 4: ['lo_fi', 'alt_country', 'americana', 'indie_pop', 'alternative', 'indie', 'folk_rock', 'acoustic', 'indie_folk', 'folk']
Cluster 5: ['post_punk', 'post_rock', 'noise_pop', 'psychedelic', 'alternative', 'lo_fi', 'indie_pop', 'indie', 'shoegaze', 'dream_pop']
Cluster 6: ['nu_jazz', 'alternative', 'instrumental', 'idm', 'experimental', 'lounge', 'ambient', 'trip_hop', 'chillout', 'downtempo']
Cluster 7: ['punk', 'metal', 'shoegaze', 'alt

In [17]:
cluster_names = {
    0: "folk / acoustic",
    1: "classic rock / blues / psychedelic",
    2: "shoegaze / dream pop / indie experimental",
    3: "art pop / neo soul / jazz",
    4: "punk / garage rock",
    5: "britpop / indie rock",
    6: "ambient / chill electronic",
    7: "goth / dark wave / post-punk"
}

In [18]:
model_df["cluster_name"] = model_df["cluster"].map(cluster_names).fillna("unknown")
model_df["cluster_name"].value_counts()

cluster_name
art pop / neo soul / jazz                    37296
punk / garage rock                            7763
shoegaze / dream pop / indie experimental     7446
britpop / indie rock                          6797
folk / acoustic                               4257
ambient / chill electronic                    3976
classic rock / blues / psychedelic            3581
goth / dark wave / post-punk                  1707
Name: count, dtype: int64

In [63]:
# model_df.head()

In [62]:
# model_df.groupby("cluster_name")["track_clean"].head(5)

In [19]:
model_df.groupby("cluster_name")["artist_clean"].value_counts().groupby(level=0).head(3)

cluster_name                               artist_clean         
ambient / chill electronic                 moby                     108
                                           massive attack           103
                                           arms and sleepers         99
art pop / neo soul / jazz                  red hot chili peppers    207
                                           david bowie              205
                                           green day                166
britpop / indie rock                       lana del rey             144
                                           beach house               98
                                           calmly                    89
classic rock / blues / psychedelic         radiohead                197
                                           placebo                  165
                                           myslovitz                156
folk / acoustic                            the cure                 215

In [20]:
model_df.to_csv("../data/processed/lastfm_music_clusters_final.csv", index=False)
model_df.to_parquet("../data/processed/lastfm_music_clusters_final.csv", index=False)


In [ ]:
# why not audio features?
# Spotify audio features were unavailable due to API limitations (403 errors), so I used Last.fm tags as a semantic representation of music, which actually captures genre and mood more directly.